# Indexes
Indexes are a DB mechanism to access stored data in a efficient way to diminish the amount of I/O operations (scans) to retrieve the data, the scan operations are:

1. Sequential Scan (seq scan): scans each table item sequentially
2. Index Scan: scans data by using index data structure and then the whole HEAP page is fetch to get whole data (retrieve more columns)
3. Index Only Scan: scans only the index data structure (only fetch columns within the index)
4. Bitman Scan: similar to index scan but creates a bit map (memory address map) of the data in a HEAP page, to avoid fetching whole HEAP page
5. TID Scan: scans the data directlr related to a tuple memory address (TID)


## No Index (Heap Table)
When no index is created then engine performs sequence scan (seq scan) which fetch all data from table to perfom the operations

In [ ]:
-- Create HEAP Table (Without Index)
SELECT * 
INTO transactionhistory_hp
FROM transactionhistory t;

-- Query test
SELECT
	p."name" AS productName,
	p.productid  AS productId,
	p.daystomanufacture AS dayToManufacture,
	th.quantity AS transactionQuantity,
	th.transactiontype AS transactionType
FROM
	product p
JOIN productdocument p2 ON p.productid = p2.productid
JOIN "document" d ON p2.documentnode = d.documentnode
JOIN transactionhistory_hp th  ON th.productid = p.productid
WHERE th.transactionid = '100018'

## Clustered Indexes
Clustered indexes specify the table to be phisically reordered based on the index.

The engine performs **index scan** which only retrieves a minimum amount of data to do the followin operations

In [ ]:
-- Copy transaction history table
SELECT * 
INTO transactionhistory_cl
FROM transactionhistory t;

-- Create PK index
CREATE UNIQUE INDEX IF NOT EXISTS PK_TransactionHistoryCL_TransactionID 
ON production.transactionhistory_cl (transactionid)


-- Create cluster
CLUSTER production.transactionhistory_cl USING PK_TransactionHistoryCL_TransactionID;

-- Query test
SELECT
	p."name" AS productName,
	p.productid  AS productId,
	p.daystomanufacture AS dayToManufacture,
	t2.quantity AS transactionQuantity,
	t2.transactiontype AS transactionType
FROM
	product p
JOIN productdocument p2 ON p.productid = p2.productid
JOIN "document" d ON p2.documentnode = d.documentnode
JOIN transactionhistory_cl t2  ON t2.productid = p.productid
WHERE t2.transactionid = '100018'


## NON-CLUSTERED Index

Non-clustered indexes are directory to map the access to data and only fetch the necessary data.

In [ ]:
-- Query test
SELECT
	p."name" AS productName,
	p.productid  AS productId,
	p.daystomanufacture AS dayToManufacture,
	t.quantity AS transactionQuantity,
	t.transactiontype AS transactionType
FROM
	product p
JOIN productdocument p2 ON p.productid = p2.productid
JOIN "document" d ON p2.documentnode = d.documentnode
JOIN transactionhistory t ON t.productid = p.productid
WHERE t.transactionid = '100018'

## Columstore Index

Some DB engines like MS SQL or Postgresql (With Citus Extension) have a data storage that allows to phisically store data in columnar structure which helps for analytics and to compress large amounts of data.

In [ ]:
-- Enable citus extension for the DB
CREATE EXTENSION IF NOT EXISTS citus;

-- Create table for columstore
CREATE TABLE production.transactionhistory_cs (
	transactionid serial4 NOT NULL,
	productid int4 NOT NULL,
	referenceorderid int4 NOT NULL,
	referenceorderlineid int4 DEFAULT 0 NOT NULL,
	transactiondate timestamp DEFAULT now() NOT NULL,
	transactiontype bpchar(1) NOT NULL,
	quantity int4 NOT NULL,
	actualcost numeric NOT NULL,
	modifieddate timestamp DEFAULT now() NOT NULL,
	CONSTRAINT "CK_TransactionHistory_CS_TransactionType" CHECK ((upper((transactiontype)::text) = ANY (ARRAY['W'::text, 'S'::text, 'P'::text]))),
	CONSTRAINT "PK_TransactionHistory_CS_TransactionID" PRIMARY KEY (transactionid)
) USING columnar;

-- Copu data into columstore
INSERT INTO production.transactionhistory_cs
SELECT * FROM production.transactionhistory;

-- Query test
SELECT
	p."name" AS productName,
	p.productid  AS productId,
	p.daystomanufacture AS dayToManufacture,
	t2.quantity AS transactionQuantity,
	t2.transactiontype AS transactionType
FROM
	production.product p
JOIN production.productdocument p2 ON p.productid = p2.productid
JOIN production."document" d ON p2.documentnode = d.documentnode
JOIN production.transactionhistory_cs t2  ON t2.productid = p.productid
WHERE t2.transactionid = '100018'

## Filtered Index

Filtered Indexes are indexes built only for a subset of data which means that only the data that matches the index criteria will be optimized

In [ ]:
CREATE INDEX IF NOT EXISTS PK_TransactionHistory_Filtered_ProductID
ON production.transactionhistory(productid)
WHERE productid <> '784';

-- Query test with index search (Fast retrieval)
SELECT
	p."name" AS productName,
	p.productid  AS productId,
	p.daystomanufacture AS dayToManufacture,
	t.quantity AS transactionQuantity,
	t.transactiontype AS transactionType
FROM
	production.product p
JOIN production.productdocument p2 ON p.productid = p2.productid
JOIN production."document" d ON p2.documentnode = d.documentnode
JOIN production.transactionhistory t ON t.productid = p.productid
WHERE t.productid = '798'

-- Query test without index search (Slow retrieval)
SELECT
	p."name" AS productName,
	p.productid  AS productId,
	p.daystomanufacture AS dayToManufacture,
	t.quantity AS transactionQuantity,
	t.transactiontype AS transactionType
FROM
	production.product p
JOIN production.productdocument p2 ON p.productid = p2.productid
JOIN production."document" d ON p2.documentnode = d.documentnode
JOIN production.transactionhistory t ON t.productid = p.productid
WHERE t.productid = '784'

In [ ]:
-- Query to search indexes
SELECT *
FROM pg_indexes
WHERE schemaname = 'production' AND tablename = 'transactionhistory_cl'
ORDER BY tablename

## Monitoring Indexes